# Breakout Room #3 — Recomendar: filtrado colaborativo por ítem y por cliente

**Objetivo:**
- Recomendar productos con la matriz del breakout anterior, de dos maneras: **por ítem** y **por cliente**.
- Entender *por qué* cada método recomienda lo que recomienda.
- Comparar costo, estabilidad y qué pasa con clientes sin historia.

Dos ideas, la misma matriz:

| | Pregunta que contesta | Cómo puntúa un producto candidato |
|---|---|---|
| **Por ítem** (item-based) | "A quien compró X, ¿qué otros productos *se parecen* a lo que ya compró?" | suma de la similitud entre el candidato y **cada producto que el cliente ya tiene** |
| **Por cliente** (user-based) | "¿Qué compraron los clientes *parecidos a mí* que yo no he comprado?" | suma de lo que compraron los **K vecinos más parecidos**, ponderada por su similitud |

Donde dice **`# TODO`** hay algo para que ustedes completen.

In [0]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
VERDE, OSCURO, NARANJA, GRIS = "#00704A", "#11241C", "#C4512F", "#9A988E"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})

orders = pd.read_csv("data/orders.csv", parse_dates=["order_date"])
customers = pd.read_csv("data/customers.csv")
CUTOFF = orders["order_date"].max() - pd.Timedelta(days=90)
train = orders[orders["order_date"] <= CUTOFF]

# La matriz del breakout anterior (1 = el cliente compró ese producto antes del corte)
M = (pd.crosstab(train["customer_id"], train["item"]) > 0).astype(int).reindex(customers["customer_id"], fill_value=0)
X = M.values.astype(float)
productos = list(M.columns)
print("Matriz:", M.shape)

## Paso 1 — las similitudes

Dos matrices de similitud coseno: **productos × productos** (29 × 29) y **clientes × clientes** (2,000 × 2,000).
Ponemos la diagonal en 0 (un producto o cliente no es "vecino" de sí mismo). Fíjense en cuánto tarda cada una.

In [0]:
t0 = time.perf_counter(); S_item = cosine_similarity(X.T); np.fill_diagonal(S_item, 0); t_item = time.perf_counter() - t0
t0 = time.perf_counter(); S_user = cosine_similarity(X);   np.fill_diagonal(S_user, 0); t_user = time.perf_counter() - t0
print(f"Matriz producto×producto: {S_item.shape}  → {S_item.size:,} celdas · {t_item*1000:.1f} ms")
print(f"Matriz cliente×cliente:   {S_user.shape}  → {S_user.size:,} celdas · {t_user*1000:.1f} ms")

## Paso 2 — recomendar por ítem

Para un cliente, el puntaje de cada producto es la suma de su similitud con **los productos que el cliente ya compró**.
Ya excluimos lo que el cliente tiene (no se recomienda lo que ya compró) y el **Agua** (ya vimos que no distingue a nadie).

In [0]:
def top_k(scores, cid, k=3):
    owned = M.loc[cid].values > 0
    s = pd.Series(scores, index=productos).where(~owned).drop("Agua", errors="ignore")
    return s.sort_values(ascending=False).head(k)

def puntaje_item(cid):
    u = M.loc[cid].values.astype(float)         # lo que el cliente compró (vector de 0/1)
    # TODO 1: puntaje de cada producto = suma de su similitud con lo que el cliente ya compró.
    # Pista: es un producto punto (matricial) entre u y S_item.
    return u @ S_item                            # puntaje: suma de similitud con lo que ya compró

for cid in (14, 62):
    print(f"Cliente {cid} compró: {', '.join(M.columns[M.loc[cid] == 1])}")
    print(top_k(puntaje_item(cid), cid).round(2).to_string(), "\n")

## Paso 3 — recomendar por cliente

Buscamos a los **K = 30 clientes más parecidos** y sumamos lo que compraron, ponderado por cuánto se parecen.

In [0]:
K_VEC = 30

def puntaje_user(cid):
    i = M.index.get_loc(cid)
    sims = S_user[i]                              # similitud de este cliente con todos los demás
    vecinos = np.argsort(-sims)[:K_VEC]           # los K más parecidos
    # TODO 2: puntaje de cada producto = suma de lo que compraron los vecinos, ponderada por su similitud.
    # Pista: sims[vecinos] (un vector de K pesos) por X[vecinos] (K filas de la matriz).
    return sims[vecinos] @ X[vecinos]              # suma ponderada de lo que compraron los vecinos

for cid in (14, 62):
    print(f"Cliente {cid}:")
    print(top_k(puntaje_user(cid), cid).round(2).to_string(), "\n")

## Paso 4 — ¿por qué recomienda eso? (explicabilidad)

Un recomendador que no se puede explicar es difícil de defender ante el negocio. Para el cliente 14, miremos el **motivo** de la primera recomendación de cada método.

In [0]:
cid = 14
rec_i = top_k(puntaje_item(cid), cid, 1).index[0]
rec_u = top_k(puntaje_user(cid), cid, 1).index[0]

# Por ítem: ¿de qué productos que ya compró viene el puntaje?
j = productos.index(rec_i)
tiene = [p for p in productos if M.loc[cid, p] == 1 and p != "Agua"]
motivo_i = pd.Series({p: S_item[productos.index(p), j] for p in tiene}).sort_values(ascending=False)
print(f"POR ÍTEM → '{rec_i}' porque se parece a lo que ya compró:")
print(motivo_i.round(2).head(4).to_string(), "\n")

# Por cliente: ¿qué vecinos lo compraron?
i = M.index.get_loc(cid)
vec = np.argsort(-S_user[i])[:K_VEC]
con_prod = [(int(M.index[v]), round(float(S_user[i, v]), 2)) for v in vec if M.iloc[v][rec_u] == 1][:4]
print(f"POR CLIENTE → '{rec_u}' porque lo compraron vecinos como (cliente, similitud): {con_prod}")
print(f"   {sum(M.iloc[v][rec_u] for v in vec)} de {K_VEC} vecinos lo compraron.")

## Paso 5 — ¿coinciden los dos métodos? ¿cuánto cuesta cada uno?

In [0]:
con_hist = M.index[M.sum(axis=1) >= 2]
col_agua = productos.index("Agua")
def mejor(sc, owned):
    s = np.where(owned > 0, -np.inf, sc); s[col_agua] = -np.inf
    return np.argsort(-s)[:3]

coinc1, coinc3 = [], []
t0 = time.perf_counter()
for cid in con_hist:
    owned = M.loc[cid].values
    ti, tu = mejor(puntaje_item(cid), owned), mejor(puntaje_user(cid), owned)
    coinc1.append(ti[0] == tu[0]); coinc3.append(len(set(ti) & set(tu)))
dt = time.perf_counter() - t0
print(f"Clientes con al menos 2 productos: {len(con_hist):,}")
print(f"Misma recomendación #1: {np.mean(coinc1):.0%}")
print(f"Coinciden en al menos 2 de las 3 primeras: {np.mean(np.array(coinc3) >= 2):.0%}")
print(f"Tiempo de recomendar a todos (los dos métodos): {dt:.1f} s")

## Paso 6 — los casos difíciles

¿Qué le recomienda cada método a un cliente con **un solo producto** y a uno con **cero**?

In [0]:
uno = M.index[M.sum(axis=1) == 1][0]
cero = M.index[M.sum(axis=1) == 0][0]
pop = pd.Series(X.sum(axis=0), index=productos)

print(f"Cliente {uno} (compró: {', '.join(M.columns[M.loc[uno] == 1])})")
print("  por ítem   :", list(top_k(puntaje_item(uno), uno).index))
print("  por cliente:", list(top_k(puntaje_user(uno), uno).index))
print(f"\nCliente {cero} (0 productos)")
print("  por ítem   :", list(top_k(puntaje_item(cero), cero).round(2).items()))
print("  por cliente:", list(top_k(puntaje_user(cero), cero).round(2).items()))
print("  (todo en 0: el método no tiene nada que ordenar)")
print("\nRespaldo común: lo más popular →", list(pop.drop("Agua").sort_values(ascending=False).head(3).index))

1. Los dos métodos, ¿recomiendan lo mismo? ¿En qué casos preferirían cada uno?

    Según los resultados del notebook, de los 1,669 clientes con al menos dos productos:
- 68% recibió la misma recomendación principal.
- 82% tuvo al menos dos coincidencias entre sus tres primeras recomendaciones.

    Preferiríamos el filtrado por ítem cuando buscamos recomendaciones basadas en productos relacionados con las compras anteriores, especialmente porque suele ser más estable y escalable.

    El filtrado por cliente sería útil cuando queremos aprovechar los patrones de compra de otros consumidores similares, permitiendo descubrir productos que el cliente todavía no conoce.

2. Tiempos y tamaños del Paso 1: si la base fuera de 2 millones de clientes y 10,000 productos, ¿cuál de las dos matrices de similitud podrían calcular y guardar de antemano?

    Preferiríamos calcular y almacenar la matriz de similitud entre productos, porque requiere considerablemente menos tiempo y menos espacio.

3. ¿Qué hacen con el cliente de 0 productos? ¿Y con un producto *nuevo* que nadie ha comprado todavía?

    Para un cliente con 0 productos, ninguno de los métodos dispone de información suficiente para calcular recomendaciones personalizadas.

    Para un producto nuevo utilizaríamos recomendaciones basadas en sus características, como categoría, ingredientes o tipo de producto, además de promocionarlo inicialmente para generar interacciones.

4. ¿cómo sabrían cuál de los dos recomienda *mejor*?


    Para determinar cuál método funciona mejor, realizaríamos una evaluación utilizando compras históricas para entrenar los modelos y compras posteriores para validar sus recomendaciones.
Compararíamos métricas como:
- Precision: proporción de los productos recomendados que realmente compró el cliente.
- Recall: proporción de las compras reales que logramos anticipar.
- NDCG: evalúa si los productos relevantes aparecen en las primeras posiciones.

    Posteriormente, podríamos realizar una prueba A/B, mostrando recomendaciones por ítem a un grupo de clientes y por cliente a otro, para comparar cuál genera mayores tasas de conversión, ventas e ingresos.

## Paso 7 — cuál recomienda mejor: evaluación con el holdout

Para responder la pregunta 4 se hace una evaluación con datos que ningún método ha visto.

- **Entrenamiento:** todo hasta el corte. De ahí salen la matriz y las similitudes que ya se usaron.
- **Holdout:** los 90 días posteriores al corte. De ahí sale la respuesta: los productos **nuevos** que cada cliente probó (que nunca había comprado antes). El Agua no cuenta, porque no se recomienda.

Se evalúan tres recomendadores: **popularidad** (lo más comprado, igual para todos), **filtrado por ítem** y **filtrado por cliente**. La popularidad es la línea base: si un método más complejo no le gana, no vale la pena usarlo.

### 7.1 Respuesta correcta de cada cliente

La matriz `NUEVO` marca con 1 los productos que el cliente probó en el holdout y que no había comprado antes del corte. Los clientes `elegibles` son los que probaron al menos un producto nuevo; solo sobre ellos se calculan las métricas.

In [0]:
holdout = orders[orders["order_date"] > CUTOFF]

H = (pd.crosstab(holdout["customer_id"], holdout["item"]) > 0).astype(int)
H = H.reindex(index=M.index, columns=productos, fill_value=0).values

NUEVO = H * (1 - X)              # lo que probó en el holdout y no había comprado antes
NUEVO[:, col_agua] = 0           # el Agua no se recomienda, no cuenta como acierto
elegibles = NUEVO.sum(axis=1) > 0

print(f"Corte: {CUTOFF.date()}")
print(f"Clientes que probaron algo nuevo en los 90 días del holdout: {elegibles.sum():,} de {len(M):,} ({elegibles.mean():.0%})")

### 7.2 Puntajes de los tres recomendadores

Cada método produce una matriz de puntajes: una fila por cliente y una columna por producto. Es el mismo cálculo de los pasos 2 y 3, pero aplicado a todos los clientes a la vez.

In [0]:
S_pop    = np.tile(X.sum(axis=0), (len(M), 1))     # popularidad: lo más comprado, igual para todos
S_por_it = X @ S_item                              # por ítem: similitud con lo que ya compró

S_por_us = np.zeros_like(X)                        # por cliente: suma ponderada de los K vecinos
for i in range(len(X)):
    vecinos = np.argsort(-S_user[i])[:K_VEC]
    S_por_us[i] = S_user[i, vecinos] @ X[vecinos]

metodos = {"Popularidad": S_pop, "Filtrado por ítem": S_por_it, "Filtrado por cliente": S_por_us}
print("Matrices de puntajes:", {n: s.shape for n, s in metodos.items()})

### 7.3 Métricas

Para cada cliente se toman las 3 recomendaciones con mejor puntaje, sin repetir lo que ya compró ni el Agua, y se comparan con lo que realmente probó en el holdout.

- **Hit rate@3:** porcentaje de clientes elegibles para los que al menos una de las 3 recomendaciones fue un producto que probaron.
- **Precisión@3:** de las 3 recomendaciones, qué fracción fueron aciertos, en promedio.
- **Recall@3:** de los productos nuevos que el cliente probó, qué fracción apareció entre las 3 recomendaciones, en promedio.
- **Cobertura:** cuántos productos distintos del catálogo recomienda el método (de 28, sin el Agua). Un método que siempre recomienda lo mismo es poco útil.

In [0]:
def top3_todos(scores, K=3):
    s = np.where(X > 0, -np.inf, scores)    # no recomendar lo ya comprado
    s[:, col_agua] = -np.inf                # ni el Agua
    return np.argsort(-s, axis=1)[:, :K]

def evaluar(scores, K=3):
    top = top3_todos(scores, K)
    aciertos = np.array([NUEVO[u, top[u]].sum() for u in range(len(top))])
    probados = NUEVO.sum(axis=1)
    return {
        "hit rate@3": (aciertos[elegibles] > 0).mean(),
        "precisión@3": (aciertos[elegibles] / K).mean(),
        "recall@3": (aciertos[elegibles] / probados[elegibles]).mean(),
        "cobertura (de 28)": len(np.unique(top)),
    }

resultados = pd.DataFrame({n: evaluar(s) for n, s in metodos.items()}).T
resultados["mejora vs popularidad"] = resultados["hit rate@3"] / resultados.loc["Popularidad", "hit rate@3"] - 1
resultados.round(3)

In [0]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
colores = [GRIS, VERDE, VERDE]
ax[0].barh(resultados.index[::-1], resultados["hit rate@3"][::-1] * 100, color=colores[::-1])
ax[0].set_xlabel("Hit rate@3 (%)"); ax[0].set_title("¿Acertaron algo nuevo?")
ax[1].barh(resultados.index[::-1], resultados["cobertura (de 28)"][::-1], color=colores[::-1])
ax[1].set_xlabel("Productos distintos recomendados"); ax[1].set_title("¿Recomiendan variedad?")
plt.tight_layout(); plt.show()

mejor = resultados["hit rate@3"].idxmax()
print(f"Mayor hit rate@3: {mejor} ({resultados.loc[mejor, 'hit rate@3']:.1%}), "
      f"{resultados.loc[mejor, 'mejora vs popularidad']:.0%} más que la popularidad.")

### 7.4 Conclusión

Los dos filtrados colaborativos superan con claridad a la popularidad: aciertan en alrededor del 60% de los clientes, frente a cerca del 35% de la línea base, y además recomiendan los 28 productos del catálogo en lugar de 13. El filtrado por ítem queda ligeramente por encima del filtrado por cliente, y es más barato y más fácil de explicar, por lo que es el candidato a producción.

Dos advertencias. Primero, los clientes sin historia reciben recomendaciones sin sentido (todos sus puntajes son 0), así que para ellos debe usarse la popularidad como respaldo. Segundo, estas métricas miden si el cliente probó el producto, no si lo probó porque se le recomendó; el efecto real en ingresos solo se conoce con una prueba A/B con grupo de control.